# Install Spark + import lib + start spark session

In [0]:
pip install --upgrade pip

In [0]:
#install findspark for Python
#!pip install -q findspark
#Install extra lib(s)
!pip install -q xlrd
!pip install -q kaggle
!pip install -q kora

#import and set environment for spark
import os

import kora
import pandas as pd

from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import *

# Check the pyspark version
# import pyspark
# print(pyspark.__version__)

In [0]:
%fs ls /user/hive/warehouse/

In [0]:
#input_path = "file:/dbfs/FileStore/table/digits.csv"
file_location = "/FileStore/tables/digits.csv"
file_type = "csv"

# CSV options
infer_schema = "true"
first_row_is_header = "true"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
data = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)

data.show()

In [0]:
from pyspark import SparkContext, SparkConf
from pyspark.ml.linalg import Vectors
from pyspark.mllib.linalg.distributed import RowMatrix

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=data.columns[1:], outputCol='features')
data_2 = assembler.setHandleInvalid("skip").transform(data)

In [0]:
display(data_2.limit(10))

In [0]:
from pyspark.ml.feature import PCA
pca = PCA(k=3, inputCol='features', outputCol='features_pca')

In [0]:
pca_model = pca.fit(data_2)

In [0]:
pca_data = pca_model.transform(data_2).select('label','features_pca')

In [0]:
pca_data.show(20,False)

In [0]:
pca_data.printSchema()

In [0]:
print(pca_model.explainedVariance)

In [0]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import ArrayType, DoubleType

def to_array(col):
    def to_array_(v):
        return v.toArray().tolist()
    # Important: asNondeterministic requires Spark 2.3 or later
    # It can be safely removed i.e.
    # return udf(to_array_, ArrayType(DoubleType()))(col)
    # but at the cost of decreased performance
    return udf(to_array_, ArrayType(DoubleType())).asNondeterministic()(col)

pca_data=(pca_data
    .withColumn("PCA", to_array(col("features_pca")))
    .select(["label"] + [col("PCA")[i] for i in range(3)]))

pca_data.show()

In [0]:
pca_data.printSchema()

In [0]:
pandasDF = pca_data.toPandas()
print(pandasDF)

In [0]:
pip install numba

In [0]:
pip install bokeh

In [0]:
import numpy as np
import numba
import pandas as pd
import scipy.special
import scipy.stats as st

# Package to perform PCA
import sklearn.datasets
import sklearn.decomposition

# BE/Bi 103 Utilities from Justin
# import bebi103

import matplotlib.pyplot as plt
import mpl_toolkits.mplot3d
import seaborn as sns
rc={'lines.linewidth': 2, 'axes.labelsize': 14, 'axes.titlesize': 14}
sns.set(rc=rc)

# Make Matplotlib plots appear inline
%matplotlib inline

import bokeh

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

classes = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
colors = ['indigo','b', 'c', 'k', 'g', 'm', 'w', 'r', 'y', 'lightgreen']

for clas, color in zip(classes, colors):
    ax.scatter(pandasDF.loc[pandasDF['label'] == clas, 'PCA[0]'],
               pandasDF.loc[pandasDF['label'] == clas, 'PCA[1]'],
               pandasDF.loc[pandasDF['label'] == clas, 'PCA[2]'],
               ) 
      
ax.set_xlabel('PCA1')
ax.set_ylabel('PCA2')
ax.set_zlabel('PCA3')

plt.show()